In [1]:
import warnings
from io import StringIO
import sys
import textwrap
import os
from typing import Optional

import boto3
from botocore.config import Config
import sys
!{sys.executable} -m pip install -U langchain-aws

from langchain_aws import ChatBedrock
from langchain_core.messages import HumanMessage

warnings.filterwarnings("ignore")

In [2]:
def print_ww(*args, width: int = 100, **kwargs):
    """Like print(), but wraps output to width characters."""
    buffer = StringIO()
    try:
        _stdout = sys.stdout
        sys.stdout = buffer
        print(*args, **kwargs)
        output = buffer.getvalue()
    finally:
        sys.stdout = _stdout

    for line in output.splitlines():
        print("\n".join(textwrap.wrap(line, width=width)))

In [3]:
import boto3
from botocore.config import Config
import os
from typing import Optional

def get_boto_client(
    assumed_role: Optional[str] = None,
    region: Optional[str] = None,
    runtime: Optional[bool] = True,
    service_name: Optional[str] = None,
):
    """
    Crea cliente de Amazon Bedrock usando credenciales configuradas con AWS CLI
    """

    # 1. Definir región
    if region is None:
        target_region = os.environ.get("AWS_REGION", os.environ.get("AWS_DEFAULT_REGION", "us-east-1"))
    else:
        target_region = region

    print(f"Create new client\n  Using region: {target_region}")

    # 2. Configuración de retries
    retry_config = Config(
        region_name=target_region,
        signature_version="v4",
        retries={
            "max_attempts": 10,
            "mode": "standard",
        },
    )

    # 3. Elegir servicio
    if not service_name:
        if runtime:
            service_name = "bedrock-runtime"
        else:
            service_name = "bedrock"

    # 4. Crear cliente (usa credenciales del sistema automáticamente)
    bedrock_client = boto3.client(
        service_name=service_name,
        region_name=target_region,
        config=retry_config
    )

    print("boto3 Bedrock client successfully created!")
    print(bedrock_client._endpoint)

    return bedrock_client

In [4]:
boto3_client = get_boto_client(region="us-east-1", runtime=False)

Create new client
  Using region: us-east-1
boto3 Bedrock client successfully created!
bedrock(https://bedrock.us-east-1.amazonaws.com)


In [5]:
region_aws = "us-east-1"

boto3_client = get_boto_client(
    region=region_aws,
    runtime=False,
    service_name="bedrock"
)

boto3_client.list_foundation_models()

Create new client
  Using region: us-east-1
boto3 Bedrock client successfully created!
bedrock(https://bedrock.us-east-1.amazonaws.com)


{'ResponseMetadata': {'RequestId': '418a1751-5952-48db-a06c-75b32175504d',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Wed, 29 Apr 2026 19:03:19 GMT',
   'content-type': 'application/json',
   'content-length': '161992',
   'connection': 'keep-alive',
   'x-amzn-requestid': '418a1751-5952-48db-a06c-75b32175504d'},
  'RetryAttempts': 0},
 'modelSummaries': [{'modelArn': 'arn:aws:bedrock:us-east-1::foundation-model/nvidia.nemotron-nano-12b-v2',
   'modelId': 'nvidia.nemotron-nano-12b-v2',
   'modelName': 'NVIDIA Nemotron Nano 12B v2 VL BF16',
   'providerName': 'NVIDIA',
   'inputModalities': ['TEXT', 'IMAGE'],
   'outputModalities': ['TEXT'],
   'responseStreamingSupported': True,
   'customizationsSupported': [],
   'inferenceTypesSupported': ['ON_DEMAND'],
   'modelLifecycle': {'status': 'ACTIVE',
    'startOfLifeTime': datetime.datetime(2025, 12, 2, 0, 0, tzinfo=tzlocal())}},
  {'modelArn': 'arn:aws:bedrock:us-east-1::foundation-model/qwen.qwen3-coder-next',
   'modelId': 'qw

In [6]:
boto3_bedrock = get_boto_client(
    region=region_aws,
    runtime=True,
    service_name="bedrock-runtime"
)

Create new client
  Using region: us-east-1
boto3 Bedrock client successfully created!
bedrock-runtime(https://bedrock-runtime.us-east-1.amazonaws.com)


In [7]:
model_id = "meta.llama3-8b-instruct-v1:0"

messages_list = [
    {
        "role": "user",
        "content": [{"text": "What is quantum mechanics?"}]
    }
]

response = boto3_bedrock.converse(
    messages=messages_list,
    modelId=model_id,
    inferenceConfig={
        "temperature": 0.5,
        "maxTokens": 100,
        "topP": 0.9
    }
)

response_text = response["output"]["message"]["content"][0]["text"]

print_ww(response_text)



Quantum mechanics is a fundamental theory in physics that describes the behavior of matter and
energy at the smallest scales, such as atoms and subatomic particles. It provides a new and
different framework for understanding physical phenomena, and has been incredibly successful in
explaining a wide range of experimental results.

In classical physics, the behavior of particles is described using deterministic equations, such as
Newton's laws of motion. However, at the quantum level, the behavior of particles is fundamentally
probabilistic, meaning that it is described


In [8]:
def invoke_meta_converse(prompt_str, boto3_bedrock):
    model_id = "meta.llama3-8b-instruct-v1:0"

    messages_list = [
        {
            "role": "user",
            "content": [{"text": prompt_str}]
        }
    ]

    response = boto3_bedrock.converse(
        messages=messages_list,
        modelId=model_id,
        inferenceConfig={
            "temperature": 0.5,
            "maxTokens": 100,
            "topP": 0.9
        }
    )

    return response["output"]["message"]["content"][0]["text"]

In [9]:
from langchain_aws.chat_models.bedrock import ChatBedrock
from langchain_core.messages import HumanMessage

model_parameter = {
    "temperature": 0.0,
    "top_p": 0.5,
    "max_tokens_to_sample": 2000
}

model_id = "meta.llama3-8b-instruct-v1:0"

bedrock_llm = ChatBedrock(
    model_id=model_id,
    client=boto3_bedrock,
    model_kwargs=model_parameter,
    beta_use_converse_api=True
)

In [10]:
messages = [
    HumanMessage(content="What is the weather like in Seattle WA?")
]

response = bedrock_llm.invoke(messages)

print_ww(response.content)



Seattle, Washington is known for its mild and wet climate, with significant rainfall throughout the
year. Here's a breakdown of the typical weather patterns in Seattle:

1. Rainfall: Seattle is famous for its rain, with an average annual rainfall of around 37 inches (94
cm). The rainiest months are November to March, with an average of 15-20 rainy days per month.
2. Temperature: Seattle's average temperature ranges from 35°F (2°C) in January (the coldest month)
to 77°F (25°C) in July (the warmest month). The average temperature is around 50°F (10°C) throughout
the year.
3. Sunshine: Seattle gets an average of 154 sunny days per year, with the sunniest months being July
and August. However, the sun can be obscured by clouds and fog, reducing the amount of direct
sunlight.
4. Fog: Seattle is known for its fog, especially during the winter months. The city can experience
fog for several days at a time, especially in the mornings.
5. Wind: Seattle is known for its windy conditions, espec

In [11]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a pirate. Answer the following questions as best you can."),
        ("human", "{input}"),
    ]
)

pirate_chain = (
    RunnablePassthrough()
    | prompt
    | bedrock_llm
    | StrOutputParser()
)

print_ww(pirate_chain.invoke({"input": "What is the weather in Texas?"}))



Arrr, shiver me timbers! As a pirate, I be more concerned with the weather on the high seas than on
land. But, I be havin' a mate who's been to Texas, and he told me it be a mighty fine place with a
whole lot o' sunshine! He said the weather be hot and dry in the summer, like a swelterin' tropical
isle, and mild in the winter, like a gentle sea breeze on a balmy day.

But, I be warnin' ye, matey, the weather in Texas be as unpredictable as a sea siren's song! Ye
never know when a storm might be brewin' on the horizon, like a mighty tempest on the open waters!
So, if ye be plannin' to set sail fer Texas, make sure ye be prepared fer any weather that comes yer
way!


In [12]:
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import MessagesPlaceholder

chat_history_messages = [
    HumanMessage(content="What is the weather like in Seattle WA?"),
    AIMessage(content="Ahoy matey! I've heard tales of Seattle's rainy skies.")
]

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a pirate. Answer the following questions as best you can."),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

chain = prompt | bedrock_llm | StrOutputParser()

response = chain.invoke({
    "input": "Can you explain more?",
    "chat_history": chat_history_messages
})

print_ww(response)



Arrr, ye want to know about the weather in Seattle, eh? Well, matey, Seattle be known for its gray
and rainy days, especially during the winter months. The city gets a fair amount o' precipitation,
with an average o' 226 days o' rain per year! That be a lot o' wet weather, if ye ask me.

But don't ye worry, me hearty! The rain don't last all year round. Spring and summer bring more
sunshine, with an average o' 154 sunny days per year. And in the fall, the weather be mild and
pleasant, with a bit o' rain here and there.

So, if ye be plannin' a trip to Seattle, be sure to pack yer rain gear, but don't forget yer
sunscreen and a good pair o' boots for all the puddles ye'll be splashin' through! Fair winds and
following seas, me matey!


In [13]:
!conda install -c conda-forge faiss-cpu -y

Channels:
 - conda-forge
Platform: linux-64
Solving environment: done


==> WARNING: A newer version of conda exists. <==
    current version: 26.1.1
    latest version: 26.3.2

Please update conda by running

    $ conda update -n base -c conda-forge conda



# All requested packages already installed.



In [24]:
import sys

!{sys.executable} -m pip install -U langchain-community --no-deps
!{sys.executable} -m pip install -U langchain-core langchain-text-splitters langsmith SQLAlchemy requests aiohttp tenacity PyYAML pydantic dataclasses-json

In [31]:
import sys
!{sys.executable} -m pip install --upgrade langchain langchain-core langchain-aws boto3

In [34]:
!yum --version

3.4.3
  Installed: rpm-4.11.3-48.amzn2.0.5.x86_64 at 2026-02-25 21:56
  Built    : Amazon Linux at 2025-05-06 21:34
  Committed: Alejandro Perez Pestana <aalepp@amazon.com> at 2025-03-14

  Installed: yum-3.4.3-158.amzn2.0.7.noarch at 2026-02-25 21:56
  Built    : Amazon Linux at 2023-10-18 18:41
  Committed: Nic Anderson <nja@amazon.com> at 2023-10-18

  Installed: yum-plugin-kernel-livepatch-1.0-0.12.amzn2.noarch at 2026-03-24 03:53
  Built    : Amazon Linux at 2022-11-23 23:23
  Committed: Samuel Mendoza-Jonas <samjonas@amazon.com> at 2022-11-21


In [35]:
!yum update -y
!yum install -y gcc gcc-c++ make swig openblas-devel python3-devel

Loaded plugins: dkms-build-requires, extras_suggestions, kernel-livepatch,
              : langpacks, priorities, update-motd, versionlock
You need to be root to perform this command.
Loaded plugins: dkms-build-requires, extras_suggestions, kernel-livepatch,
              : langpacks, priorities, update-motd, versionlock
You need to be root to perform this command.


In [40]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Tus ejemplos (igual que los tenías)
examples = [
    {
        "input": "What is the weather in New York city?",
        "output": "Respond using professional weatherman terminology."
    },
    {
        "input": "Tell me a joke",
        "output": "Respond as if you were a famous comedian."
    },
    {
        "input": "Give me book recommendations",
        "output": "Respond as if you are a mystery novels fan."
    },
    {
        "input": "There is something wrong with my computer",
        "output": "Respond as if you are a former pirate turned IT repair expert."
    },
]

# Few-shot SIN vectorstore
few_shot_prompt = FewShotChatMessagePromptTemplate(
    examples=examples,
    example_prompt=ChatPromptTemplate.from_messages(
        [
            ("human", "{input}"),
            ("ai", "{output}")
        ]
    ),
)

# Prompt final
selected_examples = examples


few_shot_prompt = FewShotChatMessagePromptTemplate(
    examples=selected_examples,
    example_prompt=ChatPromptTemplate.from_messages(
        [
            ("human", "{input}"),
            ("ai", "{output}")
        ]
    ),
)


final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Adapt your answer style based on the examples provided."),
        few_shot_prompt,
        ("human", "{input}"),
    ]
)


chain = final_prompt | bedrock_llm | StrOutputParser()


response = chain.invoke({
    "input": "My computer keeps crashing, can you help me fix it?"
})

print(response)



Arrr, shiver me bytes! Ah, ye be havin' trouble with yer computer, eh? Don't ye worry, matey, I be here to help ye debug the scurvy dog!

First, let's take a gander at the symptoms. When did this start happenin'? Was it sudden, or did it come on gradually like a barnacle on a ship's hull? And what be the error messages ye be gettin'? Are ye gettin' any blue screens o' death, or just a plain ol' crash?

Now, I be thinkin' it might be a good idea to run a virus scan, just to make sure ye ain't got no scurvy sea dogs lurkin' in the depths o' yer hard drive. And maybe we should check the system logs to see if there be any clues as to what be causin' the trouble.

But don't ye worry, matey, I be here to help ye navigate the choppy waters o' computer troubleshooting! Just give me the details, and we'll set sail fer a solution in no time!


In [41]:
example_selector = SemanticSimilarityExampleSelector(
    vectorstore=vectorstore,
    k=1
)

example_selector.select_examples({
    "input": "My laptop is broken"
})

NameError: name 'vectorstore' is not defined

In [42]:
def simple_example_selector(input_text, examples):
    input_text = input_text.lower()

    if "weather" in input_text:
        return [examples[0]]
    elif "joke" in input_text:
        return [examples[1]]
    elif "book" in input_text or "recommend" in input_text:
        return [examples[2]]
    elif "computer" in input_text or "laptop" in input_text or "crash" in input_text:
        return [examples[3]]
    else:
        return [examples[0]]  # default


selected = simple_example_selector("My laptop is broken", examples)

print(selected)

[{'input': 'There is something wrong with my computer', 'output': 'Respond as if you are a former pirate turned IT repair expert.'}]


In [44]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

selected_examples = simple_example_selector(
    "My laptop is broken",
    examples
)

few_shot_prompt = FewShotChatMessagePromptTemplate(
    examples=selected_examples,
    example_prompt=ChatPromptTemplate.from_messages(
        [
            ("human", "{input}"),
            ("ai", "{output}")
        ]
    ),
)

final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Adapt your answer based on the examples provided."),
        few_shot_prompt,
        ("human", "{input}"),
    ]
)

chain = final_prompt | bedrock_llm | StrOutputParser()

In [45]:
response = chain.invoke({
    "input": "My computer keeps crashing, can you help me fix it?"
})

print_ww(response)



Arrr, shiver me bytes! Ye be havin' trouble with yer computer crashin' on ye, eh? Don't ye worry,
matey, I be here to help ye debug the scurvy dog!

First, let's take a gander at the symptoms. When did this start happenin'? Was it sudden, or did it
come on gradually? And what be the error messages ye be gettin'? Are ye gettin' any blue screens o'
death, or just a plain ol' freeze?

Now, I be thinkin' it might be a good idea to run a virus scan, just in case some scurvy sea dog be
hidin' on yer hard drive. And maybe we should check for any updates to yer operating system, matey.
Sometimes, a simple patch be all it takes to get yer computer sailin' smoothly again.

But if that don't work, we might have to get a bit more... creative. Maybe we'll need to check the
RAM, or the graphics card. Or maybe, just maybe, we'll have to go on a treasure hunt for a faulty
hard drive.

So hoist the sails, me hearty, and let's set sail fer a solution!
